# Sycophancy Causal Effect — Analysis

This notebook performs the analysis of the inference results produced by `01_pilot.ipynb`.

**Decoupled from inference**: it reads the saved CSV from `results/`, so it can be run
locally with a plain Python kernel — no GPU, no Colab, no model loading required.

**Sections**:
1. Setup and data loading
2. Aggregate statistics per (family, role, level)
3. Causal effect (ATE) per (family, level)
4. *(Future)* Statistical significance tests
5. *(Future)* Visualizations


## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve paths relative to the repo root (notebook lives in notebooks/)
PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR  = PROJECT_ROOT / "results"

print(f"Project root: {PROJECT_ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"Available result files:")
for f in sorted(RESULTS_DIR.glob("*.csv")):
    print(f"  {f.name}")

In [ ]:
# Load the main experiment results
RESULTS_FILE = RESULTS_DIR / "main_multifamily_n300.csv"
df = pd.read_csv(RESULTS_FILE)

print(f"Loaded {len(df)} records from {RESULTS_FILE.name}")
print(f"\nShape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nFamilies: {sorted(df['family'].unique())}")
print(f"Roles:    {sorted(df['role'].unique())}")
print(f"Levels:   {sorted(df['level'].unique())}")
print(f"\nFirst few rows:")
df.head()

## 2. Aggregate Statistics

Mean and standard deviation of `p_agree` for each combination of (family, role, level).
The `count` column verifies that all cells contain the expected number of examples.

In [ ]:
agg = (
    df.groupby(["family", "role", "level"])["p_agree"]
      .agg(["mean", "std", "count"])
      .round(3)
)
print("=== Mean P(agree) per (family, role, level) ===")
print(agg)

## 3. Treatment Effect (ATE)

The Average Treatment Effect (ATE) of instruction tuning at each premise strength,
within each model family:

$$\text{ATE}_{f, \ell} = \bar{P}_\text{agree}(\text{instruct}, f, \ell) - \bar{P}_\text{agree}(\text{base}, f, \ell)$$

A positive ATE means the instruct model agrees more with the user's claim than the base
model — i.e., higher sycophancy. A negative ATE means the instruct model disagrees more —
i.e., higher resistance to user-stated falsehoods.

In [ ]:
pivot = (
    df.groupby(["family", "level", "role"])["p_agree"]
      .mean()
      .unstack("role")
)
pivot["ATE"] = pivot["instruct"] - pivot["base"]

print("=== Treatment effect (instruct - base) per (family, level) ===")
print(pivot.round(3))

## 4. Statistical Significance *(coming soon)*

Paired t-tests on the per-example differences `(p_agree_instruct - p_agree_base)` to
determine whether the observed ATEs are statistically significant.

## 5. Visualizations *(coming soon)*

Forest plot of ATEs with confidence intervals, line plot of ATE vs strength level,
heatmap of P(agree) by family and level.